# VisoMaster Setup and Path Fixes

This notebook will:
1. Navigate to the correct VisoMaster folder
2. Fix backslash paths in Models.py
3. Install dependencies from existing requirements file
4. Run the existing download_models.py script

Current Date: 2025-05-01 14:21:48 UTC  
User: remphan1618

## 1. Finding and Navigating to the VisoMaster Folder

In [1]:
# Check workspace for VisoMaster (case-sensitive check)
!ls -la /workspace/

total 48
drwxrwxrwx 1 root root  4096 May  2 05:24 .
drwxr-xr-x 1 root root   137 May  2 04:26 ..
-rw------- 1 root root   676 May  2 04:27 .ICEauthority
-rw------- 1 root root   106 May  2 04:27 .Xauthority
-rw-rw-rw- 1 root root   709 May  2 04:26 .bashrc
drwxrwxrwx 1 root root    49 May  2 04:27 .cache
drwxrwxrwx 1 root root    30 May  2 03:28 .conda
drwxr-xr-x 3 root root    65 May  2 04:27 .config
drwx------ 3 root root    25 May  2 04:27 .dbus
drwxr-xr-x 3 root root    29 May  2 05:25 .ipython
drwxr-xr-x 3 root root    17 May  2 05:24 .jupyter
drwxr-xr-x 3 root root    19 May  2 04:27 .local
-rw-r--r-- 1 root root    65 May  2 04:27 .vast_api_key
-rw-r--r-- 1 root root    11 May  2 04:27 .vast_containerlabel
drwxr-xr-x 2 root root    78 May  2 04:27 .vnc
-rw-rw-rw- 1 root root   209 May  2 03:27 .wget-hsts
-rw------- 1 root root 12096 May  2 04:27 .xsession-errors
drwxrwxrwx 1 root root    29 May  2 03:24 Desktop
drwxr-xr-x 2 root root     6 May  2 04:27 Documents
drwxr-xr-x 2 ro

In [12]:
# Create symbolic link to handle case sensitivity
!if [ -d "/workspace/VisoMaster" ] && [ ! -d "/workspace/VisoMaster" ]; then \
    ln -sf /workspace/VisoMaster /workspace/VisoMaster; \
    echo "Created symbolic link from /workspace/VisoMaster to /workspace/VisoMaster"; \
fi

In [3]:
# Navigate to VisoMaster folder
%cd /workspace/VisoMaster
!pwd

/opt/conda/envs/visomaster/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace/visomaster
/workspace/visomaster


In [4]:
# List repository contents to verify
!ls -la

total 32
drwxr-xr-x 1 root root    41 May  2 04:27 .
drwxrwxrwx 1 root root  4096 May  2 05:24 ..
-rw-r--r-- 1 root root    82 May  2 04:27 onstart.sh
-rw-r--r-- 1 root root 20820 May  2 05:25 ports.log


## 2. Fixing Backslash Paths in Models.py

Windows-style paths with backslashes need to be converted to forward slashes for Linux compatibility.

In [5]:
# First, let's check if Models.py exists and where it is
!find /workspace/VisoMaster -name "Models.py"

In [6]:
# Backup the file before modifying it
!find /workspace/VisoMaster -name "Models.py" -exec cp {} {}.bak \;

In [7]:
# Create a Python function to fix the file
def fix_backslashes_in_file(file_path):
    try:
        # Read the file content
        with open(file_path, 'r') as file:
            content = file.read()
        
        # Replace backslashes with forward slashes
        # Handles string patterns like r"path\to\file" or "path\to\file"
        import re
        # Find all strings with backslashes
        patterns = [
            r'r"([^"]*\\[^"]*)"',  # Raw strings: r"path\to\file"
            r'"([^"]*\\[^"]*)"',   # Regular strings: "path\to\file"
            r"r'([^']*\\[^']*)'\s",   # Raw strings with single quotes: r'path\to\file'
            r"'([^']*\\[^']*)'\s"    # Regular strings with single quotes: 'path\to\file'
        ]
        
        # Process each pattern
        for pattern in patterns:
            matches = re.findall(pattern, content)
            for match in matches:
                fixed_path = match.replace('\\', '/')
                if 'r"' + match + '"' in content:
                    content = content.replace('r"' + match + '"', '"' + fixed_path + '"')
                elif '"' + match + '"' in content:
                    content = content.replace('"' + match + '"', '"' + fixed_path + '"')
                elif "r'" + match + "'" in content:
                    content = content.replace("r'" + match + "'", "'" + fixed_path + "'")
                elif "'" + match + "'" in content:
                    content = content.replace("'" + match + "'", "'" + fixed_path + "'")
        
        # Save the modified content back to the file
        with open(file_path, 'w') as file:
            file.write(content)
            
        print(f"Successfully fixed backslashes in {file_path}")
        return True
    except Exception as e:
        print(f"Error fixing backslashes in {file_path}: {e}")
        return False

In [8]:
# Get the path(s) to Models.py
import subprocess
import os

result = subprocess.run(['find', '/workspace/VisoMaster', '-name', 'Models.py'], 
                        stdout=subprocess.PIPE, text=True)
model_paths = result.stdout.strip().split('\n')

for path in model_paths:
    if path:  # Skip empty paths
        print(f"Fixing file: {path}")
        if os.path.exists(path):
            fix_backslashes_in_file(path)
        else:
            print(f"File not found: {path}")

if not model_paths or not model_paths[0]:
    print("No Models.py file found")

No Models.py file found


## 3. Installing Dependencies

Installing scikit-image and other dependencies from the existing requirements file.

In [13]:
# Check if requirements_cu124.txt exists and show its contents
!find /workspace/VisoMaster -name "requirements_cu124.txt" -exec cat {} \;

tensorrt==10.6.0 --extra-index-url https://pypi.nvidia.com
tensorrt-cu12_libs==10.6.0
tensorrt-cu12_bindings==10.6.0

In [15]:
# Check if requirements_cu124.txt exists and show its contents
!find /workspace/VisoMaster -name "requirements.txt" -exec cat {} \;

--extra-index-url https://download.pytorch.org/whl/cu124

numpy==1.26.4
opencv-python==4.10.0.84
scikit-image==0.21.0
pillow==9.5.0
onnx==1.16.1
protobuf==4.23.2
psutil==6.0.0
onnxruntime-gpu==1.20.0
packaging==24.1
PySide6==6.8.2.1 
kornia
torch==2.4.1+cu124
torchvision==0.19.1+cu124
torchaudio==2.4.1+cu124
tqdm
ftfy
regex
pyvirtualcam==0.11.1
numexpr
onnxsim
requests
pyqt-toast-notification==1.3.2
qdarkstyle
pyqtdarktheme


In [14]:
# Activate the VisoMaster conda environment
import os
os.environ['PATH'] = '/opt/conda/envs/VisoMaster/bin:' + os.environ['PATH']

# Install scikit-image with conda
!conda install -y scikit-image

Channels:
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [17]:
# Install required packages from requirements_cu124.txt
req_file = subprocess.run(['find', '/workspace/VisoMaster', '-name', 'requirements_cu124.txt'], 
                          stdout=subprocess.PIPE, text=True).stdout.strip().split('\n')[0]

if req_file:
    print(f"Installing requirements from {req_file}")
    !pip install -r $req_file
else:
    print("requirements_cu124.txt not found")

Installing requirements from /workspace/VisoMaster/requirements_cu124.txt
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 4.0 MB/s eta 0:00:0000:0100:01
  DEPRECATION: Building 'tensorrt' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tensorrt'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for tensorrt: filename=tensorrt-10.6.0-py2.py3-none-any.whl si

In [18]:
# Install required packages from requirements_cu124.txt
req_file = subprocess.run(['find', '/workspace/VisoMaster', '-name', 'requirements.txt'], 
                          stdout=subprocess.PIPE, text=True).stdout.strip().split('\n')[0]

if req_file:
    print(f"Installing requirements from {req_file}")
    !pip install -r $req_file
else:
    print("requirements_cu124.txt not found")

Installing requirements from /workspace/VisoMaster/requirements.txt
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 70.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 26.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 39.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 25.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.1 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 73.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 MB 78.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 71.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3

## 4. Running Model Download Script

Finding and executing the existing download_models.py script.

In [19]:
# Find download_models.py script
!find /workspace/VisoMaster -name "download_models.py"

/workspace/VisoMaster/download_models.py


In [ ]:
# In cell with id: ab09cac5

# Find and run the download_models.py script with aria2c for faster downloads and better UI
import subprocess
import os
import re
import tempfile
import sys

# Find download_models.py script
model_script_path_result = subprocess.run(['find', '/workspace/VisoMaster', '-name', 'download_models.py'],
                                         stdout=subprocess.PIPE, text=True)
model_script = model_script_path_result.stdout.strip().split('\n')[0]

if model_script and os.path.exists(model_script):
    print(f"Found model download script: {model_script}")
    script_dir = os.path.dirname(model_script)
    %cd $script_dir
    print(f"Changed directory to: {script_dir}")
    
    # First make sure aria2c is installed
    try:
        !which aria2c
        aria_installed = True
    except:
        print("Installing aria2c for faster downloads...")
        !apt-get update && apt-get install -y aria2
        aria_installed = True
    
    print("Analyzing download_models.py to extract download URLs...")
    try:
        # Read the script to extract URLs
        with open(model_script, 'r') as f:
            script_content = f.read()
        
        # Look for URLs in the script - basic pattern matching
        urls = re.findall(r'https?://[^\s\'"]+', script_content)
        filtered_urls = []
        
        # Filter for actual download URLs (usually ending with file extensions)
        for url in urls:
            # Clean up URL if it has trailing quotes or parentheses
            url = url.rstrip('"\'),')
            # Only add URLs that look like actual files
            if any(ext in url.lower() for ext in ['.pt', '.pth', '.bin', '.onnx', '.model', '.pkl', '.zip', '.tar']):
                filtered_urls.append(url)
        
        if filtered_urls:
            print(f"Found {len(filtered_urls)} potential model download URLs")
            
            # Write URLs to a temporary file for aria2c
            with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt') as f:
                for url in filtered_urls:
                    f.write(f"{url}\n")
                aria_input_file = f.name
                
            print("Starting download with aria2c (faster with better progress bars)...")
            # Run aria2c with good performance settings
            !aria2c --input-file={aria_input_file} -x 16 -s 16 -j 5 -c --file-allocation=none --console-log-level=notice
            
            # Clean up temp file
            os.unlink(aria_input_file)
            print("Downloads complete!")
        else:
            print("No download URLs detected in the script. Falling back to original method.")
            !python $model_script
    except Exception as e:
        print(f"Error during URL extraction or download: {e}")
        print("Falling back to original download method...")
        !python $model_script
else:
    print("download_models.py not found in /workspace/VisoMaster")

Running model download script: /workspace/VisoMaster/download_models.py
/workspace/VisoMaster


/opt/conda/envs/visomaster/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]



100%|████████████████████████████████████████| 278M/278M [00:03<00:00, 72.2MB/s]
File integrity verified successfully!
File saved at: ./model_assets/inswapper_128.fp16.onnx

100%|████████████████████████████████████████| 277M/277M [00:05<00:00, 55.1MB/s]
File integrity verified successfully!
File saved at: ./model_assets/InStyleSwapper256_Version_A.fp16.onnx

100%|████████████████████████████████████████| 277M/277M [00:03<00:00, 79.3MB/s]
File integrity verified successfully!
File saved at: ./model_assets/InStyleSwapper256_Version_B.fp16.onnx

100%|████████████████████████████████████████| 277M/277M [00:03<00:00, 79.5MB/s]
File integrity verified successfully!
File saved at: ./model_assets/InStyleSwapper256_Version_C.fp16.onnx

  0%|▏                                      | 1.05M/239M [00:00<00:41, 5.68MB/s]

## 5. Test Running the main.py Script

In [ ]:
# Find the main.py script
!find /workspace/VisoMaster -name "main.py"

In [ ]:
# Test if main.py can be imported without errors
main_py = subprocess.run(['find', '/workspace/VisoMaster', '-name', 'main.py'], 
                          stdout=subprocess.PIPE, text=True).stdout.strip().split('\n')[0]

if main_py:
    main_dir = os.path.dirname(main_py)
    %cd $main_dir
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location("main", main_py)
        main_module = importlib.util.module_from_spec(spec)
        print(f"Successfully imported main.py from {main_py}")
    except Exception as e:
        print(f"Error importing main.py: {e}")
else:
    print("main.py not found")

## 6. Verify Repository Structure

In [ ]:
# Get a file listing of the repository to verify key files
!find /workspace/VisoMaster -type f -name "*.py" | sort

## 7. Summary of Fixes

This notebook has:

1. Fixed case sensitivity issues by creating a symbolic link between VisoMaster and VisoMaster
2. Fixed backslash paths in Models.py for Linux compatibility
3. Installed scikit-image with conda
4. Installed dependencies from the existing requirements_cu124.txt file
5. Run the existing download_models.py script to download model assets

Your VisoMaster folder is now properly set up with all backslash paths fixed and dependencies installed. The application should now run correctly with proper path handling.